In [ ]:
# كود المقارنة

In [ ]:
# Bootstrap و McNemar's

In [1]:
"""
================================================================================
COMPREHENSIVE MODEL COMPARISON + STATISTICAL SIGNIFICANCE ANALYSIS
Brain Tumor MRI Classification
================================================================================
هذا الملف الموحّد يحتوي على:

  PART 1 — نفس محتوى السكربت الأصلي بدون أي تعديل في المنطق:
      - النماذج السبعة (Proposed_Model + 6 baselines: CNN_SGD, ResNet18,
        DenseNet121, EfficientNetB0, VGG16, MobileNetV2)
      - الداتاست، الـ transforms، تقسيم train/val/test
      - التدريب، التقييم، Grad-CAM
      - كل مخططات المقارنة الأصلية (Loss/Acc curves, confusion matrices,
        Grad-CAM, training/inference time, accuracy, efficiency, parameters)

  PART 2 — تحليل إحصائي مضاف (لا يعيد التدريب، يستخدم تنبؤات Test Set
  المُنتَجة أصلاً أثناء نفس حلقة التدريب):
      - Bootstrap Resampling (فواصل ثقة 95% للدقة + للفروقات المزاوجة بين
        Proposed_Model وكل Baseline)
      - McNemar's Test (اختبار مزاوج على أزواج صحيح/خطأ لكل عينة اختبار،
        exact binomial عند قلة الاختلافات وإلا chi-square بتصحيح الاستمرارية)

  المخرجات الإحصائية تُحفظ في:
      model_comparison_results/statistical_analysis/
          - bootstrap_accuracy_distributions.png
          - bootstrap_delta_vs_proposed.png
          - mcnemar_pvalue_heatmap.png
          - mcnemar_pvalue_vs_proposed_bar.png
          - statistical_summary.csv
          - statistical_results.json

================================================================================
تعديلات هذه النسخة (إصلاح مشكلتَي عرض قيم p الصغيرة جداً):

  المشكلة 1 — Bootstrap p-value يظهر "p = 0":
      * السبب: p_value محسوبة كنسبة (proportion) من عدد تكرارات Bootstrap
        التي عبرت الصفر. إذا لم تعبر أي عينة الصفر إطلاقاً (وهذا متوقع
        عندما يكون الفرق بين النموذجين كبيراً وواضحاً)، فإن هذه النسبة
        تكون 0 بالضبط حسابياً، لكن هذا **لا يعني** أن قيمة p الحقيقية
        تساوي صفراً رياضياً — فقيمة p لا تساوي صفراً أبداً من الناحية
        النظرية. الأصح إحصائياً هو الإفصاح عن حد الدقة المتاح: بما أن لدينا
        N_BOOTSTRAP=10000 تكراراً، فأدق قيمة يمكن قياسها هي 1/10000 = 1e-4،
        لذا نعرض "p < 1e-4" بدلاً من "p = 0".
      * تم إضافة format_bootstrap_pvalue() وتطبيقها في عنوان كل رسم فرعي
        في bootstrap_delta_vs_proposed.png.

  المشكلة 2 — McNemar p-value يظهر "0.0e+00" في الـ heatmap، والرسم
  العمودي يقصّ كل القيم الصغيرة جداً عند نفس السقف (floor=1e-12 => كل
  الأعمدة المهمة تصبح مطابقة عند -log10(p)=12 وتفقد معناها التفاضلي):
      * السبب الجذري: 1 - scipy.stats.chi2.cdf(x, df=1) و
        scipy.stats.binom.cdf(...) يفقدان كل الدقة العشرية (float64
        underflow) عندما تكون القيمة الحقيقية أصغر من ~1e-308، فتُرجع
        0.0 بالضبط بدلاً من رقم صغير جداً غير صفري.
      * الحل: استخدام stats.chi2.logsf(statistic, df=1) و
        stats.binom.logcdf(k, n, 0.5) وهما محسوبتان في الفضاء اللوغاريتمي
        مباشرة (مستقرتان عددياً حتى عندما تكون p نفسها غير قابلة للتمثيل
        بـ float64). من هذا نحصل على log10(p) دقيقة دوماً، حتى عندما
        p الخام = 0.0 بسبب underflow.
      * الاستخدامات:
          - mcnemar_pvalue_heatmap.png: النص داخل كل خلية يعرض الآن
            "p < 1eXX" (بدلاً من "0.0e+00") عندما تحدث underflow، باستخدام
            log10_p المحسوبة بدقة.
          - mcnemar_pvalue_vs_proposed_bar.png: ارتفاع كل عمود = -log10(p)
            الحقيقية غير المقصوصة تعسفياً (لا floor صناعي بعد الآن)، مع
            سقف بصري = 50 فقط لغرض العرض (يُذكر صراحة عند تفعيله)، والنص
            فوق كل عمود يعرض "p < 1eXX" الصحيحة إحصائياً بدل قيمة p الوهمية
            الناتجة عن القص.
================================================================================

Install requirements:
    pip install torch torchvision scikit-learn seaborn matplotlib pillow scipy
================================================================================
"""

import os
import gc
import csv
import json
import time
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as transforms
import torchvision.models as tv_models

from PIL import Image
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from scipy import stats

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score
)

warnings.filterwarnings('ignore')
plt.style.use('default')
sns.set_palette('husl')

# ============================================================
# GLOBAL CONFIG
# ============================================================
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_DIR = "Brain Tumor MRI Dataset Big"
OUTPUT_DIR = Path("model_comparison_results")
OUTPUT_DIR.mkdir(exist_ok=True)

STAT_DIR = OUTPUT_DIR / "statistical_analysis"
STAT_DIR.mkdir(exist_ok=True, parents=True)

CLASS_NAMES = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES = len(CLASS_NAMES)

IMAGE_SIZE = 128
BATCH_SIZE = 96
NUM_EPOCHS = 50
EARLY_STOP_PATIENCE = 20   # إيقاف مبكر إذا لم يتحسن val_acc لعدد epochs متتالية
TRAIN_SPLIT = 0.8

N_BOOTSTRAP = 10000            # عدد عينات Bootstrap
BOOTSTRAP_CI = 0.95             # مستوى الثقة
PROPOSED_NAME = "Proposed_Model"

# النماذج التي تبقى بدون أي تعديل (بالإضافة إلى النموذج المقترح نفسه)
UNMODIFIED_BASELINES = {"CNN_SGD"}

print("=" * 70)
print(f"Using device: {DEVICE}")
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Available Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print("=" * 70)


# ============================================================
# DATASET
# ============================================================
class BrainTumorMRIDataset(Dataset):
    def __init__(self, data_dir, split_folder="Training", transform=None):
        self.data_dir = Path(data_dir)
        self.split_folder = split_folder
        self.transform = transform
        self.class_names = CLASS_NAMES
        self.class_to_idx = {name: idx for idx, name in enumerate(self.class_names)}
        self.samples = []
        self._scan_directory()

    def _scan_directory(self):
        folder = self.data_dir / self.split_folder
        seen_paths = set()
        if folder.exists():
            for class_name in self.class_names:
                class_folder = folder / class_name
                if class_folder.exists():
                    label = self.class_to_idx[class_name]
                    for img_path in class_folder.glob('*'):
                        if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                            img_path_str = str(img_path.resolve())
                            if img_path_str not in seen_paths:
                                seen_paths.add(img_path_str)
                                self.samples.append((img_path_str, label))
        else:
            print(f"Folder not found: {folder}")

    def print_class_distribution(self, title=""):
        counts = [0] * len(self.class_names)
        for _, label in self.samples:
            counts[label] += 1
        print(f"Class distribution {title}:")
        for name, c in zip(self.class_names, counts):
            print(f"  {name}: {c} images")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        try:
            image = Image.open(img_path).convert('RGB')
            if self.transform:
                image = self.transform(image)
            return image, label
        except Exception:
            black_img = Image.new('RGB', (128, 128), (0, 0, 0))
            if self.transform:
                return self.transform(black_img), label
            return torch.zeros(3, 128, 128), label


def create_transforms(image_size=128, is_training=True, augment=True):
    """
    augment=True  -> يضيف RandomHorizontalFlip / RandomRotation / ColorJitter
                     (يُستخدم فقط لـ Proposed_Model و CNN_SGD).
    augment=False -> بدون أي augmentation (يُستخدم لبقية الـ baselines).
    """
    if is_training:
        ops = [transforms.Resize((image_size, image_size))]
        if augment:
            ops += [
                transforms.RandomHorizontalFlip(p=0.4),
                transforms.RandomRotation(degrees=15),
                transforms.ColorJitter(brightness=0.2, contrast=0.2),
            ]
        ops += [
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ]
        return transforms.Compose(ops)
    return transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])


def build_train_val_datasets(data_dir, train_transform, val_transform, train_split=0.8, seed=42):
    """Reproducible 80/20 split of the Training folder (same logic as thesis scripts)."""
    base = BrainTumorMRIDataset(data_dir, split_folder="Training", transform=None)
    total = len(base)
    train_size = int(train_split * total)

    indices = list(range(total))
    np.random.seed(seed)
    np.random.shuffle(indices)

    train_idx = indices[:train_size]
    val_idx = indices[train_size:]

    train_ds = BrainTumorMRIDataset.__new__(BrainTumorMRIDataset)
    train_ds.data_dir = base.data_dir
    train_ds.split_folder = "Training (train subset)"
    train_ds.transform = train_transform
    train_ds.class_names = base.class_names
    train_ds.class_to_idx = base.class_to_idx
    train_ds.samples = [base.samples[i] for i in train_idx]

    val_ds = BrainTumorMRIDataset.__new__(BrainTumorMRIDataset)
    val_ds.data_dir = base.data_dir
    val_ds.split_folder = "Training (validation subset)"
    val_ds.transform = val_transform
    val_ds.class_names = base.class_names
    val_ds.class_to_idx = base.class_to_idx
    val_ds.samples = [base.samples[i] for i in val_idx]

    print(f"Full training pool: {total} images")
    print(f"Train subset: {len(train_ds)} | Validation subset: {len(val_ds)}")
    return train_ds, val_ds


# ============================================================
# MODEL 1: PROPOSED MODEL (patent components, unchanged)
# ============================================================
class MicroAdaptiveFeatureExtractor(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        reduced_ch = max(1, channels // reduction)
        self.spatial_conv = nn.Conv2d(channels, 1, 1)
        self.channel_avg = nn.AdaptiveAvgPool2d(1)
        self.channel_fc = nn.Conv2d(channels, reduced_ch, 1)
        self.channel_out = nn.Conv2d(reduced_ch, channels, 1)
        self.alpha = nn.Parameter(torch.tensor(0.3), requires_grad=True)

    def forward(self, x):
        spatial_weights = torch.sigmoid(self.spatial_conv(x))
        x_spatial = x * spatial_weights
        channel_pool = self.channel_avg(x_spatial)
        channel_weights = torch.sigmoid(self.channel_out(F.relu(self.channel_fc(channel_pool))))
        x_enhanced = x_spatial * channel_weights
        return x + self.alpha * x_enhanced


class MicroMultiScaleProcessor(nn.Module):
    def __init__(self, channels):
        super().__init__()
        inter = max(8, channels // 4)
        while inter % 4 != 0 and inter > 4:
            inter -= 1
        if inter < 4:
            inter = 4

        self.scale1 = nn.Conv2d(channels, inter, 1, bias=False)
        self.scale3 = nn.Conv2d(channels, inter, 3, padding=1, bias=False)

        if channels >= 8 and inter >= 4:
            groups = 1
            for g in [8, 4, 2]:
                if channels % g == 0:
                    groups = g
                    break
            self.scale3dw = nn.Conv2d(channels, inter, 3, padding=1, groups=groups, bias=False)
        else:
            self.scale3dw = nn.Conv2d(channels, inter, 3, padding=1, bias=False)

        self.fusion = nn.Conv2d(inter * 3, channels, 1, bias=False)
        self.beta = nn.Parameter(torch.tensor(0.3), requires_grad=True)

    def forward(self, x):
        s1 = F.relu(self.scale1(x))
        s3 = F.relu(self.scale3(x))
        s3dw = F.relu(self.scale3dw(x))
        combined = torch.cat([s1, s3, s3dw], dim=1)
        fused = self.fusion(combined)
        return F.relu(x + self.beta * fused)


class MicroIntelligentPooling(nn.Module):
    def __init__(self, channels):
        super().__init__()
        decision_ch = max(2, channels // 8)
        self.decision = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, decision_ch, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(decision_ch, 2, 1),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        B, C, H, W = x.shape
        decision_weights = self.decision(x)
        max_pool = F.max_pool2d(x, kernel_size=2, stride=2)
        avg_pool = F.avg_pool2d(x, kernel_size=2, stride=2)
        w1 = decision_weights[:, 0:1, :, :].view(B, 1, 1, 1)
        w2 = decision_weights[:, 1:2, :, :].view(B, 1, 1, 1)
        return w1 * max_pool + w2 * avg_pool


class UltraLightPatentBrainTumorCNN(nn.Module):
    def __init__(self, num_classes=4, base_channels=[32, 64, 128, 256]):
        super().__init__()
        c1, c2, c3, c4 = base_channels

        self.conv1 = nn.Conv2d(3, c1, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(c1)
        self.adaptive1 = MicroAdaptiveFeatureExtractor(c1, reduction=4)
        self.pool1 = MicroIntelligentPooling(c1)

        self.conv2 = nn.Conv2d(c1, c2, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(c2)
        self.multiscale2 = MicroMultiScaleProcessor(c2)
        self.pool2 = MicroIntelligentPooling(c2)

        self.conv3 = nn.Conv2d(c2, c3, 3, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(c3)
        self.adaptive3 = MicroAdaptiveFeatureExtractor(c3, reduction=8)
        self.pool3 = MicroIntelligentPooling(c3)

        self.conv4 = nn.Conv2d(c3, c4, 3, padding=1, bias=False)
        self.bn4 = nn.BatchNorm2d(c4)
        self.multiscale4 = MicroMultiScaleProcessor(c4)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(c4, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            elif isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0.)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.adaptive1(x)
        x = self.pool1(x)

        x = F.relu(self.bn2(self.conv2(x)))
        x = self.multiscale2(x)
        x = self.pool2(x)

        x = F.relu(self.bn3(self.conv3(x)))
        x = self.adaptive3(x)
        x = self.pool3(x)

        x = F.relu(self.bn4(self.conv4(x)))
        x = self.multiscale4(x)

        return self.classifier(x)


# ============================================================
# MODEL 2: CNN-SGD (deeper baseline CNN, ~2.4M params, trained with raw SGD)
# بدون أي تعديل — يبقى تماماً كما في السكربت الأصلي
# ============================================================
class SimpleCNN_SGD(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),

            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),

            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),

            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Dropout(0.3), nn.Linear(256, 384), nn.ReLU(inplace=True),
            nn.Dropout(0.2), nn.Linear(384, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


# ============================================================
# MODELS: TORCHVISION TRANSFER-LEARNING MODELS (EfficientNetB0, MobileNetV2, ...)
# ============================================================
def _safe_weights(weights_enum):
    try:
        return weights_enum.DEFAULT
    except Exception:
        return None


def _disable_inplace_relu(model):
    """
    Fix for Grad-CAM backward-hook crash on torchvision models:
    'Output 0 of BackwardHookFunctionBackward is a view and is being
    modified inplace' happens because in-place ReLUs conflict with
    register_full_backward_hook. Disabling inplace does not change
    accuracy, it only avoids the in-place autograd conflict.
    """
    for module in model.modules():
        if isinstance(module, nn.ReLU):
            module.inplace = False
        if isinstance(module, nn.ReLU6):
            module.inplace = False
    return model


def build_resnet18():
    weights = _safe_weights(tv_models.ResNet18_Weights)
    m = tv_models.resnet18(weights=weights)
    m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
    return _disable_inplace_relu(m)


def build_densenet121():
    weights = _safe_weights(tv_models.DenseNet121_Weights)
    m = tv_models.densenet121(weights=weights)
    m.classifier = nn.Linear(m.classifier.in_features, NUM_CLASSES)
    return _disable_inplace_relu(m)


def build_efficientnet_b0():
    weights = _safe_weights(tv_models.EfficientNet_B0_Weights)
    m = tv_models.efficientnet_b0(weights=weights)
    in_features = m.classifier[1].in_features
    m.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
    return _disable_inplace_relu(m)


def build_vgg16():
    weights = _safe_weights(tv_models.VGG16_Weights)
    m = tv_models.vgg16(weights=weights)
    in_features = m.classifier[6].in_features
    m.classifier[6] = nn.Linear(in_features, NUM_CLASSES)
    return _disable_inplace_relu(m)


def build_mobilenet_v2():
    weights = _safe_weights(tv_models.MobileNet_V2_Weights)
    m = tv_models.mobilenet_v2(weights=weights)
    in_features = m.classifier[1].in_features
    m.classifier[1] = nn.Linear(in_features, NUM_CLASSES)
    return _disable_inplace_relu(m)


def freeze_backbone(model, model_name):
    """
    يجمّد كل أوزان الـ feature extractor (backbone) ويترك فقط الطبقة
    التصنيفية الأخيرة قابلة للتدريب. يُطبَّق على كل الـ baselines
    ما عدا CNN_SGD (وبالطبع ما عدا Proposed_Model).
    """
    if model_name == 'ResNet18':
        for pname, p in model.named_parameters():
            if not pname.startswith('fc.'):
                p.requires_grad = False
    elif model_name == 'DenseNet121':
        for p in model.features.parameters():
            p.requires_grad = False
    elif model_name == 'EfficientNetB0':
        for p in model.features.parameters():
            p.requires_grad = False
    elif model_name == 'VGG16':
        for p in model.features.parameters():
            p.requires_grad = False
    elif model_name == 'MobileNetV2':
        for p in model.features.parameters():
            p.requires_grad = False
    return model


# ============================================================
# MODEL REGISTRY
# ============================================================
MODEL_CONFIGS = [
    {'name': 'Proposed_Model',  'builder': lambda: UltraLightPatentBrainTumorCNN(num_classes=NUM_CLASSES), 'optimizer': 'adam',    'lr': 1e-2},
    {'name': 'CNN_SGD',         'builder': lambda: SimpleCNN_SGD(num_classes=NUM_CLASSES),                 'optimizer': 'sgd_raw', 'lr': 1e-2},
    {'name': 'ResNet18',        'builder': build_resnet18,                                                 'optimizer': 'sgd_raw', 'lr': 1e-2},
    {'name': 'DenseNet121',     'builder': build_densenet121,                                               'optimizer': 'sgd_raw', 'lr': 1e-2},
    {'name': 'EfficientNetB0',  'builder': build_efficientnet_b0,                                           'optimizer': 'sgd_raw', 'lr': 1e-2},
    {'name': 'VGG16',           'builder': build_vgg16,                                                     'optimizer': 'sgd_raw', 'lr': 1e-2},
    {'name': 'MobileNetV2',     'builder': build_mobilenet_v2,                                              'optimizer': 'sgd_raw', 'lr': 1e-2},
]


# ============================================================
# HELPERS
# ============================================================
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())


def get_last_conv_layer(model):
    last_conv = None
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            last_conv = module
    return last_conv


def get_optimizer(model, name, lr):
    if name == 'sgd_raw':
        return optim.SGD(model.parameters(), lr=lr)
    return optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)


# ============================================================
# GRAD-CAM
# ============================================================
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.activations = None
        self.gradients = None
        target_layer.register_forward_hook(self._save_activation)
        target_layer.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, inp, out):
        self.activations = out.detach()

    def _save_gradient(self, module, grad_in, grad_out):
        self.gradients = grad_out[0].detach()

    def generate(self, input_tensor, class_idx=None):
        self.model.zero_grad()
        input_tensor = input_tensor.clone().detach().requires_grad_(True)
        output = self.model(input_tensor)
        if class_idx is None:
            class_idx = int(output.argmax(dim=1).item())
        score = output[0, class_idx]
        score.backward()

        gradients = self.gradients[0]
        activations = self.activations[0]
        weights = gradients.mean(dim=(1, 2))
        cam = torch.relu((weights.view(-1, 1, 1) * activations).sum(0))
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()
        return cam.cpu().numpy(), class_idx


def overlay_cam_on_image(img_np, cam):
    cam_img = Image.fromarray((cam * 255).astype(np.uint8)).resize((img_np.shape[1], img_np.shape[0]))
    cam_resized = np.array(cam_img) / 255.0
    heatmap = plt.cm.jet(cam_resized)[:, :, :3]
    overlay = 0.5 * heatmap + 0.5 * img_np
    return np.clip(overlay, 0, 1)


# ============================================================
# TRAINING / EVALUATION
# ============================================================
def train_model(model, model_name, optimizer_name, lr, train_loader, val_loader, num_epochs, patience=EARLY_STOP_PATIENCE):
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = get_optimizer(model, optimizer_name, lr)

    use_scheduler = optimizer_name != 'sgd_raw'
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs) if use_scheduler else None

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc = 0.0
    best_state = None
    epochs_no_improve = 0

    start = time.time()
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            preds = outputs.argmax(1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        train_loss = running_loss / len(train_loader)
        train_acc = 100.0 * correct / total

        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(DEVICE, non_blocking=True)
                labels = labels.to(DEVICE, non_blocking=True)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = outputs.argmax(1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)

        val_loss /= len(val_loader)
        val_acc = 100.0 * val_correct / val_total
        if scheduler is not None:
            scheduler.step()

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        print(f"[{model_name}] Epoch {epoch+1}/{num_epochs} | "
              f"Train Loss {train_loss:.4f} Acc {train_acc:.2f}% | "
              f"Val Loss {val_loss:.4f} Acc {val_acc:.2f}% | "
              f"NoImprove {epochs_no_improve}/{patience}")

        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()

        if epochs_no_improve >= patience:
            print(f"[{model_name}] Early stopping at epoch {epoch+1} "
                  f"(no improvement for {patience} consecutive epochs).")
            break

    training_time = time.time() - start
    if best_state is not None:
        model.load_state_dict(best_state)
        model = model.to(DEVICE)

    return model, history, training_time, best_val_acc


def evaluate_model(model, loader):
    """Returns targets, preds, probs (numpy) plus average per-image inference time (ms)."""
    model.eval()
    all_preds, all_targets, all_probs = [], [], []
    total_time = 0.0
    total_images = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE, non_blocking=True)

            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            t0 = time.time()
            outputs = model(images)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            total_time += time.time() - t0
            total_images += images.size(0)

            probs = F.softmax(outputs, dim=1)
            preds = probs.argmax(1)

            all_preds.append(preds.cpu())
            all_targets.append(labels)
            all_probs.append(probs.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()
    all_probs = torch.cat(all_probs).numpy()
    avg_inference_ms = (total_time / max(total_images, 1)) * 1000.0
    return all_targets, all_preds, all_probs, avg_inference_ms


# ============================================================
# PER-MODEL PLOTS
# ============================================================
def plot_training_curves(history, model_name, save_dir):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    axes[0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss', linewidth=2, markersize=4)
    axes[0].plot(epochs, history['val_loss'], 'r-s', label='Validation Loss', linewidth=2, markersize=4)
    axes[0].set_title(f'{model_name} - Loss Curve', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(epochs, history['train_acc'], 'b-o', label='Train Accuracy', linewidth=2, markersize=4)
    axes[1].plot(epochs, history['val_acc'], 'r-s', label='Validation Accuracy', linewidth=2, markersize=4)
    axes[1].set_title(f'{model_name} - Accuracy Curve', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.suptitle(f'{model_name} - Training Progress', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_dir / f'{model_name}_training_curves.png', dpi=200, bbox_inches='tight', facecolor='white')
    plt.close()


def plot_confusion_matrix_val_test(val_t, val_p, test_t, test_p, model_name, save_dir):
    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for ax, targets, preds, title in [
        (axes[0], val_t, val_p, 'Validation Set'),
        (axes[1], test_t, test_p, 'Test Set'),
    ]:
        cm = confusion_matrix(targets, preds, labels=list(range(NUM_CLASSES)))
        row_sums = cm.sum(axis=1)[:, np.newaxis]
        row_sums[row_sums == 0] = 1
        cm_norm = cm.astype('float') / row_sums
        sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                    ax=ax, cbar_kws={'label': 'Normalized Count'})
        ax.set_title(f'{model_name} - Confusion Matrix ({title})', fontsize=13, fontweight='bold')
        ax.set_xlabel('Predicted Label')
        ax.set_ylabel('True Label')

    plt.tight_layout()
    plt.savefig(save_dir / f'{model_name}_confusion_matrix.png', dpi=200, bbox_inches='tight', facecolor='white')
    plt.close()


def plot_gradcam_grid(model, dataset, model_name, save_dir):
    target_layer = get_last_conv_layer(model)
    if target_layer is None:
        print(f"No Conv2d layer found for Grad-CAM in {model_name}, skipping.")
        return

    gradcam = GradCAM(model, target_layer)

    samples = {}
    for img, label in dataset:
        if label not in samples:
            samples[label] = img
        if len(samples) == NUM_CLASSES:
            break

    if not samples:
        print(f"No samples available for Grad-CAM in {model_name}, skipping.")
        return

    class_indices = sorted(samples.keys())
    fig, axes = plt.subplots(1, len(class_indices), figsize=(5 * len(class_indices), 5))
    if len(class_indices) == 1:
        axes = [axes]

    for idx, class_idx in enumerate(class_indices):
        img_tensor = samples[class_idx].unsqueeze(0).to(DEVICE)
        cam, pred_class = gradcam.generate(img_tensor)

        img_np = samples[class_idx].cpu().numpy().transpose(1, 2, 0)
        img_np = (img_np - img_np.min()) / (img_np.max() - img_np.min() + 1e-8)
        overlay = overlay_cam_on_image(img_np, cam)

        axes[idx].imshow(overlay)
        axes[idx].set_title(f'True: {CLASS_NAMES[class_idx]}\nPred: {CLASS_NAMES[pred_class]}',
                             fontsize=11, fontweight='bold')
        axes[idx].axis('off')

    plt.suptitle(f'{model_name} - Grad-CAM Visualization', fontsize=15, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_dir / f'{model_name}_gradcam.png', dpi=200, bbox_inches='tight', facecolor='white')
    plt.close()


# ============================================================
# CROSS-MODEL COMPARISON PLOTS (original)
# ============================================================
def plot_training_inference_comparison(results, save_dir):
    names = [r['name'] for r in results]
    train_times_min = [r['training_time'] / 60 for r in results]
    inference_ms = [r['inference_time_ms'] for r in results]

    x = np.arange(len(names))
    width = 0.35

    fig, ax1 = plt.subplots(figsize=(16, 8))
    bars1 = ax1.bar(x - width / 2, train_times_min, width, label='Training Time (min)',
                     color='#4ECDC4', edgecolor='black')
    ax1.set_ylabel('Training Time (minutes)', fontsize=13, color='#2C7873')
    ax1.set_xticks(x)
    ax1.set_xticklabels(names, rotation=30, ha='right')

    ax2 = ax1.twinx()
    bars2 = ax2.bar(x + width / 2, inference_ms, width, label='Inference Time (ms/image)',
                     color='#FF6B6B', edgecolor='black')
    ax2.set_ylabel('Inference Time (ms per image)', fontsize=13, color='#B03A2E')

    for bar, val in zip(bars1, train_times_min):
        ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{val:.1f}',
                  ha='center', va='bottom', fontsize=9, fontweight='bold')
    for bar, val in zip(bars2, inference_ms):
        ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{val:.2f}',
                  ha='center', va='bottom', fontsize=9, fontweight='bold')

    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right', fontsize=11)

    plt.title('Training Time vs Inference Time - Full Model Comparison', fontsize=16, fontweight='bold')
    ax1.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(save_dir / '01_training_vs_inference_time_comparison.png', dpi=250, bbox_inches='tight', facecolor='white')
    plt.close()


def plot_accuracy_comparison(results, save_dir):
    names = [r['name'] for r in results]
    val_acc = [r['val_accuracy'] * 100 for r in results]
    test_acc = [r['test_accuracy'] * 100 for r in results]

    x = np.arange(len(names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(16, 8))
    b1 = ax.bar(x - width / 2, val_acc, width, label='Validation Accuracy', color='#4ECDC4', edgecolor='black')
    b2 = ax.bar(x + width / 2, test_acc, width, label='Test Accuracy', color='#FF6B6B', edgecolor='black')

    for bars in [b1, b2]:
        for bar in bars:
            h = bar.get_height()
            ax.text(bar.get_x() + bar.get_width() / 2, h + 0.3, f'{h:.2f}%',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(names, rotation=30, ha='right')
    ax.set_ylabel('Accuracy (%)', fontsize=13)
    ax.set_ylim(0, 105)
    ax.set_title('Validation vs Test Accuracy - All Models', fontsize=16, fontweight='bold')
    ax.legend(fontsize=12)
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(save_dir / '02_accuracy_comparison_val_test.png', dpi=250, bbox_inches='tight', facecolor='white')
    plt.close()


def plot_efficiency_comparison(results, save_dir):
    names = [r['name'] for r in results]
    efficiency = [(r['test_accuracy'] * 100) / (r['params'] / 1e6) for r in results]
    colors = plt.cm.viridis(np.linspace(0, 0.9, len(names)))

    fig, ax = plt.subplots(figsize=(16, 8))
    bars = ax.bar(names, efficiency, color=colors, edgecolor='black')

    for bar, val in zip(bars, efficiency):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{val:.2f}',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_ylabel('Efficiency = Test Accuracy(%) / Parameters(M)', fontsize=12)
    ax.set_title('Model Efficiency Comparison', fontsize=16, fontweight='bold')
    ax.set_xticklabels(names, rotation=30, ha='right')
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(save_dir / '03_efficiency_comparison.png', dpi=250, bbox_inches='tight', facecolor='white')
    plt.close()


def plot_parameters_comparison(results, save_dir):
    names = [r['name'] for r in results]
    params_m = [r['params'] / 1e6 for r in results]

    fig, ax = plt.subplots(figsize=(16, 8))
    bars = ax.bar(names, params_m, color='#96CEB4', edgecolor='black')

    for bar, val in zip(bars, params_m):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f'{val:.2f}M',
                ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_ylabel('Parameters (Millions, log scale)', fontsize=13)
    ax.set_yscale('log')
    ax.set_title('Model Size Comparison (Total Parameters)', fontsize=16, fontweight='bold')
    ax.set_xticklabels(names, rotation=30, ha='right')
    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(save_dir / '04_parameters_comparison.png', dpi=250, bbox_inches='tight', facecolor='white')
    plt.close()


def save_summary_table(results, save_dir):
    with open(save_dir / '00_summary_table.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Model', 'Val Accuracy (%)', 'Test Accuracy (%)', 'Parameters',
                          'Training Time (min)', 'Inference Time (ms/img)', 'Efficiency (Acc%/M)'])
        for r in results:
            eff = (r['test_accuracy'] * 100) / (r['params'] / 1e6)
            writer.writerow([
                r['name'],
                f"{r['val_accuracy']*100:.2f}",
                f"{r['test_accuracy']*100:.2f}",
                r['params'],
                f"{r['training_time']/60:.2f}",
                f"{r['inference_time_ms']:.3f}",
                f"{eff:.2f}"
            ])
    with open(save_dir / 'comparison_results.json', 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2)
    print(f"Summary table saved to: {save_dir / '00_summary_table.csv'}")


# ================================================================
# PART 2 — STATISTICAL ANALYSIS (Bootstrap + McNemar's Test)
# ================================================================

# ----------------------------------------------------------------
# NEW: helpers لعرض قيم p الصغيرة جداً بشكل صحيح إحصائياً
# ----------------------------------------------------------------
def format_pvalue_scientific(p_value, log10_p=None, min_display_log10=-300):
    """
    يُنسّق قيمة p للعرض النصي بطريقة صحيحة إحصائياً:
    - قيمة p لا تساوي صفراً رياضياً أبداً، لذا لا نعرض أبداً "p = 0" أو
      "0.0e+00".
    - إذا كانت p_value > 0 فعلياً (لم يحدث float underflow): تُعرض بصيغة
      علمية عادية، أو "p < X" إذا كانت صغيرة جداً (<1e-4).
    - إذا حدث underflow إلى 0.0 (شائع جداً مع chi-square لعدد كبير من
      الأزواج المتضاربة)، نستخدم log10_p المحسوبة بطريقة مستقرة عددياً
      (عبر logsf/logcdf) لعرض حد أعلى واقعي: "p < 1eXX" بدلاً من قيمة
      وهمية.
    """
    if p_value > 0:
        if p_value < 1e-4:
            return f"p < {p_value:.1e}"
        return f"p = {p_value:.4g}"
    if log10_p is not None and np.isfinite(log10_p):
        bound_exponent = int(np.floor(log10_p))
        bound_exponent = max(bound_exponent, min_display_log10)
        return f"p < 1e{bound_exponent}"
    return f"p < 1e{min_display_log10}"


def format_bootstrap_pvalue(p_value, n_bootstrap=N_BOOTSTRAP):
    """
    قيمة p من Bootstrap تُحسب كنسبة (proportion) من عدد التكرارات، لذا أدق
    قيمة ممكن قياسها هي 1/n_bootstrap. عدم عبور أي عينة للصفر لا يعني
    "p=0" رياضياً، بل يعني أن p أصغر من دقة القياس المتاحة بهذا العدد من
    التكرارات، فنعرض "p < resolution" بدلاً من "p = 0".
    """
    resolution = 1.0 / n_bootstrap
    if p_value <= 0:
        return f"p < {resolution:.1e}"
    if p_value < 1e-4:
        return f"p < {p_value:.1e}"
    return f"p = {p_value:.4g}"


def bootstrap_accuracy(correct_vec, n_bootstrap=N_BOOTSTRAP, seed=SEED):
    """correct_vec: مصفوفة 0/1 (1=تنبؤ صحيح) بطول عدد عينات الاختبار."""
    rng = np.random.default_rng(seed)
    n = len(correct_vec)
    idx = rng.integers(0, n, size=(n_bootstrap, n))
    return correct_vec[idx].mean(axis=1)


def bootstrap_ci(boot_values, ci=BOOTSTRAP_CI):
    alpha = 1 - ci
    lower = np.percentile(boot_values, 100 * (alpha / 2))
    upper = np.percentile(boot_values, 100 * (1 - alpha / 2))
    return lower, upper


def bootstrap_paired_delta(correct_a, correct_b, n_bootstrap=N_BOOTSTRAP, seed=SEED):
    """
    فرق الدقة (a - b) عبر Bootstrap مزاوج (نفس فهرس العينات لكلا النموذجين في
    كل تكرار). يُرجع: (boot_deltas, ci_lower, ci_upper, p_value_two_sided)

    ملاحظة: p_value هنا محسوبة كنسبة تكرارات Bootstrap. إذا كانت 0.0
    بالضبط (لم يعبر أي تكرار الصفر)، فهذا لا يعني رياضياً p=0 — استخدم
    format_bootstrap_pvalue() عند العرض بدلاً من طباعة p_value مباشرة.
    """
    rng = np.random.default_rng(seed)
    n = len(correct_a)
    idx = rng.integers(0, n, size=(n_bootstrap, n))
    acc_a = correct_a[idx].mean(axis=1)
    acc_b = correct_b[idx].mean(axis=1)
    deltas = acc_a - acc_b

    lower, upper = bootstrap_ci(deltas)
    p_left = np.mean(deltas <= 0)
    p_right = np.mean(deltas >= 0)
    p_value = min(2 * min(p_left, p_right), 1.0)

    return deltas, lower, upper, p_value


def mcnemar_test(correct_a, correct_b):
    """
    اختبار McNemar المزاوج بين نموذجين على نفس عينات الاختبار.
    جدول الطوارئ 2x2:
        [[both_correct,     a_only_correct],
         [b_only_correct,   both_wrong]]
    - Exact binomial إذا كان عدد الأزواج المتضاربة < 25
    - وإلا chi-square بتصحيح الاستمرارية

    إصلاح الاستقرار العددي:
    عند استخدام chi-square لعدد كبير من الأزواج المتضاربة، أو exact
    binomial لإحصائية كبيرة، قد تكون قيمة p الحقيقية أصغر من أن يمثّلها
    float64 (~1e-308)، فتُرجع الدوال القياسية (chi2.cdf / binom.cdf) صفراً
    بالضبط (underflow) رغم أن p ليست صفراً رياضياً أبداً. لتفادي هذا،
    نحسب log10(p) مباشرة في الفضاء اللوغاريتمي عبر chi2.logsf و
    binom.logcdf، وهما مستقرتان عددياً حتى في هذه الحالات المتطرفة. هذا
    يسمح لاحقاً بعرض "p < 1eXX" الصحيحة بدل "p = 0" أو "0.0e+00".
    """
    both_correct = int(np.sum((correct_a == 1) & (correct_b == 1)))
    both_wrong = int(np.sum((correct_a == 0) & (correct_b == 0)))
    a_only = int(np.sum((correct_a == 1) & (correct_b == 0)))
    b_only = int(np.sum((correct_a == 0) & (correct_b == 1)))

    n_discordant = a_only + b_only
    table = [[both_correct, a_only], [b_only, both_wrong]]

    if n_discordant == 0:
        return {'table': table, 'statistic': 0.0, 'p_value': 1.0, 'log10_p': 0.0,
                'method': 'no_discordant_pairs', 'n_discordant': 0}

    if n_discordant < 25:
        k = min(a_only, b_only)
        # exact binomial two-sided، محسوبة في الفضاء اللوغاريتمي لتفادي أي فقدان دقة
        log_cdf = stats.binom.logcdf(k, n_discordant, 0.5)
        log_p_two_sided = min(np.log(2.0) + log_cdf, 0.0)  # log(p) لا يتجاوز log(1)=0
        log10_p = log_p_two_sided / np.log(10)
        p_value = float(min(np.exp(log_p_two_sided), 1.0))
        statistic = float(k)
        method = 'exact_binomial'
    else:
        statistic = (abs(a_only - b_only) - 1) ** 2 / n_discordant
        logsf = stats.chi2.logsf(statistic, df=1)   # log(p) مستقر عددياً حتى لو p تحته صفر float64
        log10_p = logsf / np.log(10)
        p_value = float(np.exp(logsf))
        method = 'chi2_continuity_corrected'

    return {'table': table, 'statistic': float(statistic), 'p_value': p_value,
            'log10_p': float(log10_p), 'method': method, 'n_discordant': n_discordant}


def plot_bootstrap_accuracy_distributions(boot_data, save_dir):
    """boot_data: dict[name] -> (boot_accs, point_acc, ci_low, ci_high)"""
    names = list(boot_data.keys())
    n = len(names)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.5 * rows))
    axes = np.array(axes).reshape(-1)

    for i, name in enumerate(names):
        boot_accs, point_acc, lo, hi = boot_data[name]
        ax = axes[i]
        ax.hist(boot_accs * 100, bins=50, color='#4ECDC4', edgecolor='black', alpha=0.8)
        ax.axvline(point_acc * 100, color='black', linestyle='-', linewidth=2, label=f'Test Acc = {point_acc*100:.2f}%')
        ax.axvline(lo * 100, color='#B03A2E', linestyle='--', linewidth=1.5)
        ax.axvline(hi * 100, color='#B03A2E', linestyle='--', linewidth=1.5,
                   label=f'95% CI [{lo*100:.2f}%, {hi*100:.2f}%]')
        ax.set_title(name, fontsize=12, fontweight='bold')
        ax.set_xlabel('Bootstrap Test Accuracy (%)')
        ax.set_ylabel('Frequency')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    for j in range(len(names), len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'Bootstrap Distributions of Test Accuracy (N={N_BOOTSTRAP} resamples)',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_dir / 'bootstrap_accuracy_distributions.png', dpi=220, bbox_inches='tight', facecolor='white')
    plt.close()


def plot_bootstrap_delta_vs_proposed(delta_data, save_dir):
    """delta_data: dict[baseline_name] -> (deltas, lo, hi, p_value)"""
    names = list(delta_data.keys())
    n = len(names)
    cols = 3
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4.5 * rows))
    axes = np.array(axes).reshape(-1)

    for i, name in enumerate(names):
        deltas, lo, hi, p_value = delta_data[name]
        ax = axes[i]
        ax.hist(deltas * 100, bins=50, color='#96CEB4', edgecolor='black', alpha=0.85)
        ax.axvline(0, color='black', linestyle='-', linewidth=1.5, label='No difference')
        ax.axvline(lo * 100, color='#B03A2E', linestyle='--', linewidth=1.5)
        ax.axvline(hi * 100, color='#B03A2E', linestyle='--', linewidth=1.5,
                   label=f'95% CI [{lo*100:.2f}, {hi*100:.2f}]')
        sig = 'significant' if p_value < 0.05 else 'not significant'
        # NEW: استخدام format_bootstrap_pvalue بدلاً من "p = {p_value:.4g}" الخام،
        # حتى لا تظهر "p = 0" عندما لم يعبر أي تكرار Bootstrap الصفر.
        p_text = format_bootstrap_pvalue(p_value)
        ax.set_title(f'{PROPOSED_NAME} - {name}\n{p_text} ({sig})',
                     fontsize=11, fontweight='bold')
        ax.set_xlabel('Δ Accuracy (%)')
        ax.set_ylabel('Frequency')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

    for j in range(len(names), len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'Bootstrap Paired Difference in Accuracy: {PROPOSED_NAME} vs Baselines',
                 fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_dir / 'bootstrap_delta_vs_proposed.png', dpi=220, bbox_inches='tight', facecolor='white')
    plt.close()


# ----------------------------------------------------------------
# FIXED: plot_mcnemar_heatmap
# ----------------------------------------------------------------
# يستقبل الآن log10_matrix بالإضافة إلى pvalue_matrix، لأن النص المعروض
# في كل خلية يجب أن يعتمد على log10(p) المحسوبة بشكل مستقر عددياً وليس
# على p_value الخام فقط (والتي قد تكون 0.0 بالضبط بسبب float underflow).
# هذا يحل مشكلة ظهور "0.0e+00" غير الصحيحة إحصائياً.
# ----------------------------------------------------------------
def plot_mcnemar_heatmap(names, pvalue_matrix, log10_matrix, save_dir):
    n = len(names)
    fig, ax = plt.subplots(figsize=(1.35 * n + 3.5, 1.1 * n + 3))

    vmin, vmax = 0.0, 0.10
    cmap = sns.color_palette("rocket_r", as_cmap=True)   # مقياس ألوان احترافي
    norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)

    diag_mask = np.eye(n, dtype=bool)
    plot_matrix = np.clip(pvalue_matrix, vmin, vmax)

    sns.heatmap(
        plot_matrix, cmap=cmap, vmin=vmin, vmax=vmax,
        xticklabels=names, yticklabels=names, mask=diag_mask,
        ax=ax, cbar_kws={'label': "McNemar's p-value (clipped at 0.10)", 'shrink': 0.85},
        linewidths=1.2, linecolor='white', square=True, annot=False
    )

    # تلوين الخلايا القطرية بلون محايد داكن مع علامة توضيحية
    for i in range(n):
        ax.add_patch(mpatches.Rectangle((i, i), 1, 1, facecolor='#333333', edgecolor='white', linewidth=1.2))
        ax.text(i + 0.5, i + 0.5, '—', ha='center', va='center', color='white', fontsize=13, fontweight='bold')

    # كتابة القيم يدوياً مع اختيار لون نص يضمن تبايناً واضحاً دائماً،
    # وباستخدام format_pvalue_scientific لتفادي عرض "0.0e+00"
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            true_val = pvalue_matrix[i, j]
            log10_val = log10_matrix[i, j]
            clipped_val = plot_matrix[i, j]
            rgba = cmap(norm(clipped_val))
            luminance = 0.299 * rgba[0] + 0.587 * rgba[1] + 0.114 * rgba[2]
            text_color = 'white' if luminance < 0.55 else '#111111'

            p_text = format_pvalue_scientific(true_val, log10_val)
            # تبسيط النص داخل الخلية (بدون تكرار "p")
            label = p_text.replace('p < ', '<').replace('p = ', '')
            is_significant = (true_val > 0 and true_val < 0.05) or (true_val <= 0 and np.isfinite(log10_val) and log10_val < np.log10(0.05))
            weight = 'bold' if is_significant else 'normal'

            ax.text(j + 0.5, i + 0.5, label, ha='center', va='center',
                     color=text_color, fontsize=10, fontweight=weight)

    ax.set_title("McNemar's Test — Pairwise p-values (Test Set)\n"
                 "(darker/warmer = more statistically significant, p < 0.05)",
                 fontsize=13, fontweight='bold', pad=14)
    ax.set_xticklabels(names, rotation=35, ha='right', fontsize=10)
    ax.set_yticklabels(names, rotation=0, fontsize=10)

    plt.tight_layout()
    plt.savefig(save_dir / 'mcnemar_pvalue_heatmap.png', dpi=220, bbox_inches='tight', facecolor='white')
    plt.close()


# ----------------------------------------------------------------
# FIXED: plot_mcnemar_vs_proposed_bar
# ----------------------------------------------------------------
# التعديل الأساسي: تم إزالة "floor = 1e-12" الذي كان يقصّ كل قيم p
# الصغيرة جداً عند نفس السقف (-log10(p)=12)، مما يجعل كل الأعمدة المهمة
# متطابقة الارتفاع وتفقد التمايز فيما بينها.
#
# بدلاً من ذلك: نستخدم log10_p المحسوبة مسبقاً في mcnemar_test() عبر
# logsf/logcdf (فضاء لوغاريتمي مستقر عددياً)، وهي دقيقة حتى لو p الخام قد
# underflow-ت إلى 0.0. هذا يعطي الآن ارتفاع أعمدة حقيقياً ومتفاوتاً يعكس
# درجات الدلالة الإحصائية المختلفة فعلياً بين الأزواج.
#
# سقف العرض (DISPLAY_CAP) هنا هو تجميلي بصري فقط (لمنع عمود واحد شاذ من
# طغيانه على البقية)، ويُفصح عنه صراحة في التسمية عند تفعيله، ولا يُستخدم
# أبداً كأساس لحساب قيمة p المعروضة في النص.
# ----------------------------------------------------------------
def plot_mcnemar_vs_proposed_bar(mcnemar_vs_proposed, save_dir):
    names = list(mcnemar_vs_proposed.keys())
    p_values = [mcnemar_vs_proposed[n]['p_value'] for n in names]
    log10_p_values = [mcnemar_vs_proposed[n]['log10_p'] for n in names]

    neg_log_p_true = [-v for v in log10_p_values]

    DISPLAY_CAP = 50.0
    neg_log_p_display = [min(v, DISPLAY_CAP) for v in neg_log_p_true]

    alpha_line = -np.log10(0.05)
    colors = ['#C0392B' if (p < 0.05 or (p <= 0 and lp < np.log10(0.05))) else '#2E86AB'
              for p, lp in zip(p_values, log10_p_values)]

    fig, ax = plt.subplots(figsize=(13, 7.5))
    bars = ax.bar(names, neg_log_p_display, color=colors, edgecolor='black', linewidth=1.3, width=0.6)

    ax.axhline(alpha_line, color='#222222', linestyle='--', linewidth=1.8,
               label=f'α = 0.05  (-log10(p) = {alpha_line:.2f})')

    top = max(neg_log_p_display) if neg_log_p_display else 1.0
    offset = top * 0.03 if top > 0 else 0.05

    for bar, true_val, p_true, log10_true in zip(bars, neg_log_p_true, p_values, log10_p_values):
        p_text = format_pvalue_scientific(p_true, log10_true)
        capped_note = '\n(bar capped for display)' if true_val > DISPLAY_CAP else ''
        label = f'{p_text}{capped_note}'
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + offset, label,
                 ha='center', va='bottom', fontsize=9, fontweight='bold', color='#111111')

    ax.set_ylabel(f'-log10(p-value)  [capped at {DISPLAY_CAP:.0f} for display only — labels show true value]\n'
                  f'(taller bar = stronger statistical significance)', fontsize=11.5)
    ax.set_title(f"McNemar's Test: {PROPOSED_NAME} vs Each Baseline (Test Set)",
                 fontsize=15, fontweight='bold', pad=12)
    ax.set_xticklabels(names, rotation=30, ha='right', fontsize=11)
    ax.set_ylim(0, top * 1.3 if top > 0 else 1.0)

    sig_patch = mpatches.Patch(color='#C0392B', label='Significant (p < 0.05)')
    nonsig_patch = mpatches.Patch(color='#2E86AB', label='Not significant (p ≥ 0.05)')
    alpha_handle = ax.get_legend_handles_labels()[0][0]
    ax.legend(handles=[sig_patch, nonsig_patch, alpha_handle],
              labels=['Significant (p < 0.05)', 'Not significant (p ≥ 0.05)', 'α = 0.05 threshold'],
              fontsize=10.5, loc='upper right')

    ax.grid(alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig(save_dir / 'mcnemar_pvalue_vs_proposed_bar.png', dpi=220, bbox_inches='tight', facecolor='white')
    plt.close()


def save_statistical_summary(model_results, delta_data, mcnemar_vs_proposed, names, save_dir):
    with open(save_dir / 'statistical_summary.csv', 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow([
            'Model', 'Test Accuracy (%)', 'Bootstrap CI Low (%)', 'Bootstrap CI High (%)',
            f'Δ Acc vs {PROPOSED_NAME} (%)', 'Bootstrap Δ CI Low (%)', 'Bootstrap Δ CI High (%)',
            'Bootstrap p-value (Δ, display)', "McNemar's statistic",
            "McNemar's p-value (display)", "McNemar's log10(p)", "McNemar's method"
        ])
        for name in names:
            r = model_results[name]
            if name == PROPOSED_NAME:
                writer.writerow([name, f"{r['test_acc']*100:.2f}", f"{r['ci_low']*100:.2f}",
                                  f"{r['ci_high']*100:.2f}", '-', '-', '-', '-', '-', '-', '-', '-'])
            else:
                d = delta_data[name]
                m = mcnemar_vs_proposed[name]
                boot_p_text = format_bootstrap_pvalue(d[3])
                mcnemar_p_text = format_pvalue_scientific(m['p_value'], m['log10_p'])
                writer.writerow([
                    name, f"{r['test_acc']*100:.2f}", f"{r['ci_low']*100:.2f}", f"{r['ci_high']*100:.2f}",
                    f"{np.mean(d[0])*100:.2f}", f"{d[1]*100:.2f}", f"{d[2]*100:.2f}", boot_p_text,
                    f"{m['statistic']:.4f}", mcnemar_p_text, f"{m['log10_p']:.4f}", m['method']
                ])
    print(f"Statistical summary saved to: {save_dir / 'statistical_summary.csv'}")


def run_statistical_analysis(correct_vectors, test_accs, names):
    """يُشغَّل مرة واحدة بعد الانتهاء من تدريب/تقييم كل النماذج، باستخدام
    متجهات (صحيح/خطأ) المُجمَّعة أصلاً من evaluate_model على Test Set — بدون
    أي إعادة تدريب أو إعادة استدلال."""
    print("\n" + "=" * 70)
    print("RUNNING BOOTSTRAP RESAMPLING")
    print("=" * 70)

    boot_data = {}
    model_results = {}
    for name in names:
        correct = correct_vectors[name]
        boot_accs = bootstrap_accuracy(correct)
        lo, hi = bootstrap_ci(boot_accs)
        boot_data[name] = (boot_accs, test_accs[name], lo, hi)
        model_results[name] = {'test_acc': test_accs[name], 'ci_low': lo, 'ci_high': hi}
        print(f"[{name}] Test Acc = {test_accs[name]*100:.2f}%  "
              f"95% Bootstrap CI = [{lo*100:.2f}%, {hi*100:.2f}%]")

    baseline_names = [n for n in names if n != PROPOSED_NAME]
    proposed_correct = correct_vectors[PROPOSED_NAME]

    delta_data = {}
    for name in baseline_names:
        deltas, lo, hi, p_value = bootstrap_paired_delta(proposed_correct, correct_vectors[name])
        delta_data[name] = (deltas, lo, hi, p_value)
        p_text = format_bootstrap_pvalue(p_value)
        print(f"[{PROPOSED_NAME} vs {name}] Δ Acc = {np.mean(deltas)*100:+.2f}%  "
              f"95% CI = [{lo*100:+.2f}%, {hi*100:+.2f}%]  bootstrap {p_text}")

    print("\n" + "=" * 70)
    print("RUNNING McNEMAR'S TEST")
    print("=" * 70)

    pvalue_matrix = np.ones((len(names), len(names)))
    log10_matrix = np.zeros((len(names), len(names)))
    mcnemar_full = {}
    for i, name_a in enumerate(names):
        for j, name_b in enumerate(names):
            if i == j:
                continue
            res = mcnemar_test(correct_vectors[name_a], correct_vectors[name_b])
            pvalue_matrix[i, j] = res['p_value']
            log10_matrix[i, j] = res['log10_p']
            mcnemar_full[f"{name_a}_vs_{name_b}"] = res

    mcnemar_vs_proposed = {}
    for name in baseline_names:
        res = mcnemar_test(proposed_correct, correct_vectors[name])
        mcnemar_vs_proposed[name] = res
        p_text = format_pvalue_scientific(res['p_value'], res['log10_p'])
        print(f"[{PROPOSED_NAME} vs {name}] statistic={res['statistic']:.4f}  "
              f"{p_text}  method={res['method']}  "
              f"discordant_pairs={res['n_discordant']}")

    print("\n" + "=" * 70)
    print("GENERATING STATISTICAL PLOTS")
    print("=" * 70)

    plot_bootstrap_accuracy_distributions(boot_data, STAT_DIR)
    plot_bootstrap_delta_vs_proposed(delta_data, STAT_DIR)
    plot_mcnemar_heatmap(names, pvalue_matrix, log10_matrix, STAT_DIR)
    plot_mcnemar_vs_proposed_bar(mcnemar_vs_proposed, STAT_DIR)
    save_statistical_summary(model_results, delta_data, mcnemar_vs_proposed, names, STAT_DIR)

    json_results = {
        'test_accuracy': test_accs,
        'bootstrap_ci': {n: {'low': model_results[n]['ci_low'], 'high': model_results[n]['ci_high']}
                          for n in names},
        'bootstrap_delta_vs_proposed': {
            n: {'mean_delta': float(np.mean(delta_data[n][0])),
                'ci_low': float(delta_data[n][1]), 'ci_high': float(delta_data[n][2]),
                'p_value': float(delta_data[n][3]),
                'p_value_display': format_bootstrap_pvalue(delta_data[n][3])}
            for n in baseline_names
        },
        'mcnemar_vs_proposed': mcnemar_vs_proposed,
        'mcnemar_full_pairwise': mcnemar_full,
        'n_bootstrap': N_BOOTSTRAP,
        'confidence_level': BOOTSTRAP_CI,
    }
    with open(STAT_DIR / 'statistical_results.json', 'w', encoding='utf-8') as f:
        json.dump(json_results, f, indent=2)

    print("\n" + "=" * 70)
    print("STATISTICAL ANALYSIS COMPLETE")
    print(f"Results saved under: {STAT_DIR.resolve()}")
    print("=" * 70)


# ============================================================
# MAIN
# ============================================================
def main():
    print("=" * 70)
    print("COMPREHENSIVE MODEL COMPARISON + STATISTICAL ANALYSIS")
    print("Proposed Model (Adam) vs CNN_SGD/ResNet18/DenseNet121/EfficientNetB0/")
    print("VGG16/MobileNetV2 (all raw SGD, no optimizer tricks)")
    print("=" * 70)

    if not Path(DATA_DIR).exists():
        print(f"Data directory not found: {DATA_DIR}")
        return

    eval_transform = create_transforms(IMAGE_SIZE, is_training=False)

    # Proposed_Model و CNN_SGD: يبقيان مع augmentation كما في الأصل
    train_transform_augmented = create_transforms(IMAGE_SIZE, is_training=True, augment=True)
    train_dataset_augmented, val_dataset = build_train_val_datasets(
        DATA_DIR, train_transform_augmented, eval_transform, TRAIN_SPLIT, SEED
    )

    # بقية الـ baselines: بدون augmentation
    train_transform_plain = create_transforms(IMAGE_SIZE, is_training=True, augment=False)
    train_dataset_plain, _ = build_train_val_datasets(
        DATA_DIR, train_transform_plain, eval_transform, TRAIN_SPLIT, SEED
    )

    test_dataset = BrainTumorMRIDataset(DATA_DIR, split_folder="Testing", transform=eval_transform)

    train_dataset_augmented.print_class_distribution("(Train - augmented: Proposed_Model & CNN_SGD)")
    val_dataset.print_class_distribution("(Validation)")
    test_dataset.print_class_distribution("(Test)")

    pin_memory = DEVICE.type == 'cuda'
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=0, pin_memory=pin_memory)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=0, pin_memory=pin_memory)

    results = []
    correct_vectors = {}   # name -> مصفوفة 0/1 على Test Set (لاستخدامها في التحليل الإحصائي)
    test_accs = {}

    for cfg in MODEL_CONFIGS:
        name = cfg['name']
        print("\n" + "=" * 70)
        print(f"TRAINING MODEL: {name}  (optimizer={cfg['optimizer']}, lr={cfg['lr']})")
        print("=" * 70)

        model_dir = OUTPUT_DIR / name
        model_dir.mkdir(exist_ok=True)

        # اختيار الداتاسِت المناسب: augmentation فقط لـ Proposed_Model و CNN_SGD
        keep_unmodified = (name == PROPOSED_NAME) or (name in UNMODIFIED_BASELINES)
        train_ds = train_dataset_augmented if keep_unmodified else train_dataset_plain
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                   num_workers=0, pin_memory=pin_memory, drop_last=True)

        model = cfg['builder']()

        # تجميد الـ backbone لكل الـ baselines ما عدا CNN_SGD (والنموذج المقترح)
        if not keep_unmodified:
            model = freeze_backbone(model, name)

        params = count_parameters(model)
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"Parameters: {params:,} (trainable: {trainable_params:,})")

        model, history, training_time, best_val_acc_pct = train_model(
            model, name, cfg['optimizer'], cfg['lr'], train_loader, val_loader, NUM_EPOCHS,
            patience=EARLY_STOP_PATIENCE
        )

        plot_training_curves(history, name, model_dir)

        print(f"Evaluating {name} on Validation set...")
        val_t, val_p, val_probs, val_inf_ms = evaluate_model(model, val_loader)
        print(f"Evaluating {name} on Test set...")
        test_t, test_p, test_probs, test_inf_ms = evaluate_model(model, test_loader)

        plot_confusion_matrix_val_test(val_t, val_p, test_t, test_p, name, model_dir)

        try:
            plot_gradcam_grid(model, val_dataset, name, model_dir)
        except Exception as e:
            print(f"Grad-CAM failed for {name}: {e}")

        val_acc = accuracy_score(val_t, val_p)
        test_acc = accuracy_score(test_t, test_p)

        # حفظ متجه (صحيح/خطأ) على Test Set لاستخدامه لاحقاً في Bootstrap/McNemar
        correct_vectors[name] = (test_t == test_p).astype(np.int32)
        test_accs[name] = float(test_acc)

        print(f"\n[{name}] Test Classification Report:")
        print(classification_report(test_t, test_p, target_names=CLASS_NAMES, digits=3))

        results.append({
            'name': name,
            'params': params,
            'training_time': training_time,
            'inference_time_ms': test_inf_ms,
            'val_accuracy': val_acc,
            'test_accuracy': test_acc,
        })

        del model
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
        gc.collect()

    print("\n" + "=" * 70)
    print("GENERATING CROSS-MODEL COMPARISON PLOTS")
    print("=" * 70)

    plot_training_inference_comparison(results, OUTPUT_DIR)
    plot_accuracy_comparison(results, OUTPUT_DIR)
    plot_efficiency_comparison(results, OUTPUT_DIR)
    plot_parameters_comparison(results, OUTPUT_DIR)
    save_summary_table(results, OUTPUT_DIR)

    # -------------------------------------------------------------
    # PART 2: التحليل الإحصائي (Bootstrap + McNemar's Test)
    # يستخدم متجهات (صحيح/خطأ) المُجمَّعة أعلاه دون أي إعادة تدريب
    # -------------------------------------------------------------
    names = [cfg['name'] for cfg in MODEL_CONFIGS]
    run_statistical_analysis(correct_vectors, test_accs, names)

    print("\n" + "=" * 70)
    print("ALL MODELS TRAINED, COMPARED, AND STATISTICALLY ANALYZED SUCCESSFULLY")
    print(f"Results saved under: {OUTPUT_DIR.resolve()}")
    print("=" * 70)


if __name__ == "__main__":
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
        gc.collect()
    main()

Using device: cuda
GPU: NVIDIA GeForce RTX 5080 Laptop GPU
Available Memory: 15.9 GB
COMPREHENSIVE MODEL COMPARISON + STATISTICAL ANALYSIS
Proposed Model (Adam) vs CNN_SGD/ResNet18/DenseNet121/EfficientNetB0/
VGG16/MobileNetV2 (all raw SGD, no optimizer tricks)
Full training pool: 9650 images
Train subset: 7720 | Validation subset: 1930
Full training pool: 9650 images
Train subset: 7720 | Validation subset: 1930
Class distribution (Train - augmented: Proposed_Model & CNN_SGD):
  glioma: 2411 images
  meningioma: 1742 images
  notumor: 1558 images
  pituitary: 2009 images
Class distribution (Validation):
  glioma: 607 images
  meningioma: 441 images
  notumor: 387 images
  pituitary: 495 images
Class distribution (Test):
  glioma: 755 images
  meningioma: 546 images
  notumor: 487 images
  pituitary: 626 images

TRAINING MODEL: Proposed_Model  (optimizer=adam, lr=0.01)
Parameters: 659,228 (trainable: 659,228)
[Proposed_Model] Epoch 1/50 | Train Loss 1.3229 Acc 42.15% | Val Loss 1.1322 A